In [1]:
import numpy as np
from Scripts.Optimization.optimization import Optimizer

opt = Optimizer()
opt.independent_properties_calc()

opt.active_constraints = {
    "deflection": True,
    "stress": True,
    "frequency": True,
    "natural frequency": False,
    "energy": False,
    "damage": False,
}

opt.design_variables_initial_guess = {
    "arm_diameter": 50.0,
    "arm_thickness": 2.5,
    "strut_diameter": 50.0,
    "strut_thickness": 2.5,
    "strut_distance": 475,
    "hub_radius": 400,
    "hub_flange_thickness": 2,
    "hub_web_thickness": 2
}

opt.fixed_design_variables = {
    "arm_diameter": False,
    "arm_thickness": False,
    "strut_diameter": False,
    "strut_thickness": False,
    "strut_distance": False,
    "hub_radius": False,
    "hub_flange_thickness": False,
    "hub_web_thickness": False
}
opt.constraints_constants["allowable_deflection"] = 10
opt.simulation_settings["sample_count"] = 10


res = opt.run_opt()

x = np.array(res.x)
x_expanded = list(opt.reform_dvs(x).values())

cons_res_opt, cons_labels_opt = opt.test_results(x)

dv_keys = list(opt.design_variables_initial_guess.keys())

print("===== Optimized =====")
print("design variables")
for i in range(len(dv_keys)):
    print(f"   {dv_keys[i]}: {x_expanded[i]}")

print("constraints")
for i in range(len(cons_res_opt)):
    print(f"   {cons_labels_opt[i]}: {cons_res_opt[i]}")

print(f"mass w/ payload: {res.fun * 1000} kg")

print("OPT_OUTPUT")
opt.test_results(x, detailed_output=True)

print("Initial guess opt actual:")
print(res.x)

TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
import matplotlib.pyplot as plt
import Scripts.Finite_Elements.drone_fea as fea
import Scripts.unit_conversions as uc

## FEA Model Segment ##
fea_model_seg = opt.fea.beam_system

uav_fea_model_full = fea.DroneFEA(opt.drone_geometry)
uav_fea_model_full.create_drone_nodes()
uav_fea_model_full.create_drone_beams()
uav_fea_model_full.boundary_conditions()
uav_fea_model_full.solve_fea()

fea_model_full = uav_fea_model_full.beam_system

## Freuency Analysis ##
fea_model_seg.solve_natural_frequencies()
fea_model_full.solve_natural_frequencies()

nat_fre_hz = fea_model_seg.natural_frequencies * uc.rad_per_s_to_Hz
full_nat_fre_hz = fea_model_full.natural_frequencies * uc.rad_per_s_to_Hz

print(f"Full 50mm disp:")
print(f"First 10 Natural Frequencies: {full_nat_fre_hz[:30]} Hz")

print("Segment 50mm disp:")
print(f"First 10 Natural Frequencies: {nat_fre_hz[:30]} Hz")


# Max Frequency #

drone_power = opt.drone_power_module
drone_power.freq_trans_calc(opt.parameters["propeller_max_RPM"])
max_freq = np.max(drone_power.frequency)
min_freq = np.min(drone_power.frequency[15:1:-15])
print(f"Prop Frequency Range: {min_freq * uc.rad_per_s_to_Hz}-{max_freq * uc.rad_per_s_to_Hz} Hz")

## Frequency Plotting ##
n_samples = 100
frequency_sample = np.linspace(min_freq, max_freq, n_samples)



max_amps = []
force = opt.parameters["gravity"] * (opt.parameters["payload"] / opt.parameters["blade_num"])
for freq in frequency_sample:
    test_freq_fea = fea_model_full
    test_freq_fea.init_dynamic_forces(freq)
    for i in range(7):
        node = f"outer_node_{i}"
        test_freq_fea.add_dynamic_harmonic_force(np.array([0, 0, 1*(-1)**i, 0, 0, 0]), node)
    test_freq_fea.solve_dynamic_harmonic(0,0)
    max_amps.append(max(test_freq_fea.mag_displacements_amplitude))

plt.figure(figsize=(10, 5))
plt.plot(frequency_sample * uc.rad_per_s_to_Hz, max_amps)
plt.title("Maximum Magnitude Displacement Amplitude vs Frequency")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Max Displacement Amplitude per Unit Force (mm/N)")
plt.grid(True)
plt.yscale('log')
plt.show()